# EDA 004 · Dilution training set

**The question.** Will this company issue many new shares soon?

When a company sells new shares, every existing share becomes a smaller slice of the company:
that is *dilution*. It is common in small, cash-burning companies (biotech, early-stage tech)
that pay their bills by selling stock.

**One row** is a company at a month end (2012 – June 2022). **The answer**, `dilution`, is 1 if
its shares outstanding (adjusted for splits, as first filed) rise by 10% or more within the next
182 days, else 0; missing if no new share count was filed in that time. It is 1 in about one row
in ten (the *base rate*). The model (007) gives a chance; to be useful it must sort the rows so
that the high-chance groups really dilute more often than 10%.

Measures used: **AUC** (area under the ROC curve): the chance that a random diluting row gets a
higher score than a random non-diluting row; 0.5 is guessing, 1.0 is perfect. **Lift in the top
decile**: how many times the base rate the answer occurs among the 10% highest-scored rows.
**Brier score**: the mean squared error of a chance; for always saying the base rate p it is
p × (1 − p).

**How this notebook works**

- **What it does.** It reads one training set from the private export, keeps the development
  rows, and goes through it section by section: what is in it, who is covered, what is missing,
  what is extreme, what the answer looks like, how well the simple baseline already does, which
  features relate to the answer, which features repeat each other, and whether anything leaks
  from the future. It ends with written findings and advice for the model.
- **Why.** A model is only as good as its data. Problems found here are cheap to fix; found after
  training, they mean training again. The findings also tell the model builder where to start.
- **It changes nothing.** The notebook only reads. A problem that needs fixing is fixed in the
  pipeline (`src/sentinel/`), with tests, so that it applies to every year alike (the holdout
  included) and to the daily scores; it is never patched here.
- **Cleaning already applied by the pipeline** (decided on 30 October 2026, `docs/decisions.md`):
  V2-22, a foreign filer's market value is blank (its US price is often an ADR's while its share
  count is of ordinary shares, so the product would be wrong); V2-23, a share count typed in
  wrongly (zero or below, or 100 times away from the same filing's weighted-average count or,
  without one, from the company's last accepted count) is left out, judged only by what was known
  when it was filed; V2-24, an insider trade line worth more than the whole company has no dollar
  value (a typo in its price or shares). Section 5 checks that these hold.
- **Privacy.** The export holds prices licensed from Tiingo. Outputs are stripped when the
  notebook is committed (the repository is public); the written findings quote only aggregates.
- **Run it** with one command, under the machine's memory cap (about 30 seconds):
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/<this notebook>`

**Words used below**

- **Row**: one company on one date, with everything known about it at that day's close (the
  *features*) and what happened next (the *answer*).
- **Eligible**: the rows a model may learn from: the company filed something with the SEC in the
  last 12 months, is not a fund or blank-check company, its price is at least \$1, it trades at
  least \$10,000 a day, and it has at least 63 days of prices. Per question: price-only stocks
  need not file for volatility; foreign filers are left out of dilution.
- **Development rows**: rows up to 2022 with an answer. Rows from 2023 on are the sealed holdout
  and are never looked at here.
- **Test years**: 2019, 2020, 2021, 2022. Each is predicted by a model trained only on earlier
  years ("walk-forward").
- **Purge**: a training row whose answer window reaches into the test year is dropped, so the
  model never trains on days it is tested on.
- **Spearman correlation (ρ)**: how well two columns rank alike, from −1 to +1; 0 means no order in
  common. It ignores scale, so it is safe with long tails.
- **Decile**: one of ten equal-sized groups after sorting by a feature (decile 1 = the lowest 10%).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentinel import eda

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.4g}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
TEST_YEARS = eda.TEST_YEARS

## 1. Load

**What:** read the file, keep the development rows (dated 2022 or earlier), and count rows,
companies and dates; check that each (company, date) appears once. Then `ok` = the rows a model
will learn from: eligible and with an answer. **Why:** every later section measures relations on
`ok`, as the model will see them; the counts are the scale of everything that follows.

In [ ]:
SET, ANSWER = "dilution", "dilution"
df = eda.load(SET)
FEATURES = eda.features(df)
print(f"{len(df):,} rows, {df['cik'].nunique():,} companies, {df.shape[1]} columns, "
      f"{len(FEATURES)} features")
print("dates:", df["as_of"].min().date(), "to", df["as_of"].max().date())
print("duplicate (cik, as_of) keys:", int(df.duplicated(["cik", "as_of"]).sum()))
ELIG = eda.ELIGIBLE[SET]  # this question's eligibility (V2-26, V2-27)
ok = df[df[ELIG] & df[ANSWER].notna()].copy()  # what the model learns from
print(f"eligible with an answer: {len(ok):,} rows ({len(ok) / len(df):.1%}), "
      f"{ok['cik'].nunique():,} companies")

## 2. Columns by family

**What:** sort the about 100 features into families (price, trading, size, financials, filing
events, insider, short volume, macro) and list what is not a feature (identifiers, flags, the
answer, dates). **Why:** families make 100 columns readable, show which kinds of information the
model has, and make sure nothing that is not a feature (the answer, its window) is used as one.

In [ ]:
fam = pd.Series({c: eda.family(c) for c in FEATURES}, name="family")
display(fam.value_counts().rename("features").to_frame())
display(fam.groupby(fam).apply(lambda s: ", ".join(s.index)).rename("columns").to_frame())
others = [c for c in df.columns if c not in FEATURES]
print("not features:", ", ".join(others))
display(df.dtypes.astype(str).value_counts().rename("columns").to_frame())

## 3. Coverage

Who is in the data, per year and per kind of company. *Coverage level*: `full` (files with the
SEC and has prices), `foreign` (a foreign filer: 20-F or 40-F reports), `price_only` (trades in
the US, files nothing periodic). Size bands by market value: micro below \$300 M, small to
\$2 B, mid to \$10 B, large above; `unknown` where market value is missing.

In [ ]:
year = df["as_of"].dt.year
by_year = eda.coverage_table(df, year, "dilution", ELIG)
display(by_year)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
pd.DataFrame({"eligible": df[df[ELIG]].groupby(year[df[ELIG]]).size(),
              "not eligible": df[~df[ELIG]].groupby(year[~df[ELIG]]).size()}
             ).plot.bar(stacked=True, ax=ax[0], title="rows per year")
by_year["companies"].plot.bar(ax=ax[1], title="companies per year")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["coverage", "filer_type", "foreign_filer", "sector"]:
    display(eda.coverage_table(df, col, "dilution", ELIG).sort_values("rows", ascending=False))
df["size_band"] = eda.size_band(df["market_cap"])
ok["size_band"] = eda.size_band(ok["market_cap"])
display(eda.coverage_table(df, "size_band", "dilution", ELIG))

**Why rows are not eligible.** A row can fail several of the five rules at once. Each question
then has its own flag (decided 30 October 2026): volatility also admits price-only stocks that
pass the price rules (V2-26: volatility needs only prices); dilution leaves out foreign filers
(V2-27: most have no answer); the results move uses the general flag. The two tables after the
reasons compare the general flag with this question's. **Why it
matters:** the model never sees these rows, so whatever they have in common (tiny, untraded,
not filing) is outside what it can forecast. `no reason found` must be 0: otherwise the flag and
the rules disagree.

In [ ]:
display(eda.eligibility_reasons(df))
print(f"this question uses `{ELIG}`:")
display(pd.crosstab(df["eligible"], df[ELIG], margins=True))
display(pd.crosstab(df["coverage"], df[ELIG]))

## 4. Missing values

Share of eligible rows with an answer where each feature is missing. A missing value is not
an error when the fact does not exist for that company (constitution II: missing, never zero):
foreign filers do not file Form 4 or 8-K; companies without financial filings have no ratios; a
ratio with a zero denominator is left empty.

In [ ]:
miss = eda.missing_table(ok, FEATURES)
print(f"features with no missing values: {(miss['missing'] == 0).sum()}; "
      f"over 50% missing: {(miss['missing'] > 0.5).sum()}")
display(miss[miss["missing"] > 0])
display(miss.groupby("family")["missing"].agg(["mean", "max"]).sort_values("mean"))

**Where the gaps come from.** The 15 most-missing features by filer type, coverage level and
year. **Why:** a gap explained by the kind of company (a foreign filer has no Form 4; a company
that does no research has no R&D) is information, not an error; a gap that grows or shrinks by
year would show a change in the data, not in the companies.

In [ ]:
top = miss.index[:15].tolist()
display(ok.groupby("filer_type")[top].apply(lambda g: g.isna().mean()).T)
display(ok.groupby("coverage")[top].apply(lambda g: g.isna().mean()).T)
display(eda.missing_by_year(ok, top))

## 5. Impossible and extreme values

`infinite`: values of ±∞ (a division by zero that slipped through); `negative`: values below
zero where that is impossible (counts, prices, volatilities, amounts). Then each feature's spread:
the 1st percentile (p1), median and 99th percentile (p99) against its minimum and maximum: a
maximum far beyond p99 means a long tail or a data error.

In [ ]:
ext = eda.extremes_table(ok, FEATURES)
bad = ext[(ext["infinite"] > 0) | (ext["negative"] > 0)]
print("features with infinite or impossible negative values:", len(bad))
display(bad)
spread = ext.assign(max_over_p99=ext["max"] / ext["p99"].where(ext["p99"] > 0))
display(spread.sort_values("max_over_p99", ascending=False).head(25))

**Suspect values.** Specific checks behind the long tails above. **Why:** a long tail can be
real (a stock that rose tenfold) or a data error; these checks count the known kinds of error.
The first rows check the pipeline's cleaning rules (V2-22 to V2-24): they should be 0 or close
to it. A foreign filer's market value
should be missing (its share count is of ordinary shares while its US price is often an ADR's,
risk_params v5); no US company was worth \$5 trillion before 2023 (Apple's peak was about \$3
trillion); a 12-month insider total above \$100 billion is a typo in a filing.

In [ ]:
checks = {
    "foreign filer with a market value": ok["foreign_filer"] & ok["market_cap"].notna(),
    "market value 0": ok["market_cap"] == 0,
    "market value above 5 trillion dollars": ok["market_cap"] > 5e12,
    "shares outstanding above 50 billion": ok["shares_outstanding"] > 5e10,
    "one-month return above +1,000%": ok["ret_1m"] > 10,
    "12-month return above +2,000%": ok["ret_12_1"] > 20,
    "insider sales above 100 billion dollars in 12 months": ok["sell_value_12m"] > 1e11,
    "insider buys above 10 billion dollars in 12 months": ok["buy_value_12m"] > 1e10,
    "cash above total assets": ok["cash_to_assets"] > 1,
    "negative leverage": ok["leverage"] < 0,
    "negative book value (book_to_market < 0)": ok["book_to_market"] < 0,
    "results overdue by more than a year": ok["days_to_next_results"] < -365,
}
display(pd.DataFrame({k: {"rows": int(v.sum()), "companies": ok.loc[v, "cik"].nunique(),
                          "share": v.mean()} for k, v in checks.items()}).T)

In [ ]:
print("answer values:", ok[ANSWER].value_counts(dropna=False).to_dict())
print("eligible rows without an answer (no new share count within 182 days):",
      int((df["eligible"] & df[ANSWER].isna()).sum()))
display(df[df["eligible"]].groupby("filer_type")[ANSWER].apply(lambda s: s.isna().mean())
        .rename("eligible rows without an answer").to_frame())

## 6. The answer: base rates

How often companies dilute, overall and by year, sector, size, filer type and coverage.

In [ ]:
p = ok[ANSWER].mean()
print(f"base rate: {p:.2%} of {len(ok):,} rows; Brier score of always saying it: {p * (1 - p):.4f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ok.groupby(ok["as_of"].dt.year)[ANSWER].mean().plot.bar(ax=ax[0], title="base rate by year")
ok.groupby("size_band")[ANSWER].mean().sort_values().plot.barh(ax=ax[1], title="base rate by size")
plt.tight_layout(); plt.show()
for col in ["sector", "size_band", "filer_type", "coverage", "loss_making"]:
    display(ok.groupby(col, dropna=False)[ANSWER].agg(["count", "mean"]).sort_values("mean"))
print("companies that diluted at least once:", f"{ok.groupby('cik')[ANSWER].max().mean():.1%}")

## 7. The naive baseline and simple rules

The baseline is the base rate (the same chance for everyone: AUC 0.5). Simple rules a person
might use, scored on the test years: loss-making with less than a year of cash; any offering
filed in the last 12 months; shares grew 10% or more over the last 12 months (dilution tends to
repeat).

In [ ]:
rules = {
    "loss-making, cash runway < 1 year": ((ok["loss_making"] == 1) & (ok["cash_runway_years"] < 1)).astype(float),
    "offering filed in 12 months": (ok["n_offering_12m"] > 0).astype(float),
    "shelf effective in 12 months": (ok["n_shelf_effective_12m"] > 0).astype(float),
    "shares up 10%+ in 12 months": (ok["share_growth_12m"] >= 0.10).astype(float),
}
rows = {}
for name, flag in rules.items():
    hit = ok.loc[flag == 1, ANSWER]
    rows[name] = {"rows flagged": int(flag.sum()), "share flagged": flag.mean(),
                  "rate if flagged": hit.mean(), "rate if not": ok.loc[flag == 0, ANSWER].mean(),
                  "AUC": eda.auc(ok[ANSWER], flag)}
display(pd.DataFrame(rows).T)
test = ok[ok["as_of"].dt.year.isin(TEST_YEARS)]
display(pd.DataFrame({y: {name: eda.auc(g[ANSWER], rules[name].loc[g.index]) for name in rules}
                      for y, g in test.groupby(test["as_of"].dt.year)}))

## 8. Feature–answer relations

For a yes/no answer, each feature's **AUC** (direction-free: below 0.5 means the feature's low
values go with dilution; |AUC − 0.5| is the strength) and its **top-decile lift**, plus the
Spearman ρ used in the other notebooks.

In [ ]:
ranks = eda.rank_table(ok, FEATURES, ANSWER)
auc = {}
for c in FEATURES:
    x = ok[c]
    if x.notna().sum() > 1000 and x.nunique() > 1:
        a = eda.auc(ok[ANSWER], x)
        auc[c] = {"AUC": a, "strength": abs(a - 0.5),
                  "lift top decile": eda.top_decile_lift(ok[ANSWER], x if a >= 0.5 else -x),
                  "present": x.notna().mean()}
auc = pd.DataFrame(auc).T.sort_values("strength", ascending=False)
auc.insert(0, "family", [eda.family(c) for c in auc.index])
display(auc)
display(auc.groupby("family")["strength"].max().sort_values().rename("strongest |AUC-0.5|").to_frame())

**The dilution rate by decile** of the ten strongest features, against the base rate (dashed).
**Why:** shows where in a feature's range the risk sits, for example only in the top tenth.

In [ ]:
top10 = auc.index[:10].tolist()
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=True)
for ax, col in zip(axes.flat, top10):
    d = eda.decile_table(ok, col, ANSWER)
    ax.plot(d.index.astype(int), d["mean"], marker="o"); ax.axhline(ok[ANSWER].mean(), ls="--", c="grey")
    ax.set_title(col, fontsize=9); ax.set_xticks(range(1, 11))
plt.suptitle("dilution rate by feature decile (dashed: base rate)"); plt.tight_layout(); plt.show()

**Stability over time:** each strong feature's AUC within each year. **Why:** a relation that
holds every year is one a model can rely on in the test years and in daily use.

In [ ]:
top15 = auc.index[:15].tolist()
yearly = pd.DataFrame({y: {c: eda.auc(g[ANSWER], g[c]) if g[c].notna().sum() > 100 else np.nan
                           for c in top15}
                       for y, g in ok.groupby(ok["as_of"].dt.year)})
display(yearly)
side = np.sign(auc.loc[top15, "AUC"] - 0.5)
display(pd.DataFrame({"pooled AUC": auc.loc[top15, "AUC"], "min": yearly.min(axis=1),
                      "max": yearly.max(axis=1),
                      "years on the other side of 0.5": (np.sign(yearly - 0.5).ne(side, axis=0)
                                                         & yearly.notna()).sum(axis=1)}))

**Share-count anomalies.** The answer compares share counts, so errors in the counts become
false answers. Checks: very large jumps (a 10-for-1 count change looks like a split or a unit
error, not a share sale); rows where the company split its stock during the window; and the
cover-page versus weighted-average share source.

In [ ]:
g = ok["share_growth_12m"]
display(g.describe(percentiles=[0.01, 0.5, 0.9, 0.99, 0.999]).to_frame())
for t in [1, 3, 9]:
    print(f"share_growth_12m above {t + 1}x: {(g > t).sum():,} rows")
split_next = ok.groupby("cik")["cum_split_after"].shift(-6)
changed = (split_next.notna() & (split_next != ok["cum_split_after"]))
print(f"rows with a split in the next 6 months: {changed.sum():,}; "
      f"their dilution rate: {ok.loc[changed, ANSWER].mean():.2%} vs {ok.loc[~changed, ANSWER].mean():.2%}")

## 9. Redundant features

Pairs of features that rank rows almost identically (|ρ| ≥ 0.9, on a fixed sample of 200,000
rows). Tree models cope with them, but they split importance between the pair and make the model
harder to explain; linear models need one of each pair dropped.

In [ ]:
pairs = eda.redundant_pairs(ok, FEATURES)
print(len(pairs), "pairs")
display(pairs)

## 10. Leakage and purging

Checks: every window ends after the row date; no feature separates the answer implausibly well
(AUC above 0.95 or below 0.05); the purge before each test year.

In [ ]:
gap = (ok["window_end"] - ok["as_of"]).dt.days
print("windows ending on or before the row date:", int((gap <= 0).sum()))
display(gap.describe(percentiles=[0.01, 0.5, 0.99]).rename("days from row to window end").to_frame())
print("features with AUC >= 0.95 or <= 0.05:", auc.index[(auc["AUC"] >= 0.95) | (auc["AUC"] <= 0.05)].tolist())
display(eda.purge_table(ok, "window_end"))

## 11. Findings

*Export of 30 October 2026, after the cleaning rules V2-22 to V2-24, with this question's
eligibility `eligible_dilution` (V2-27: foreign filers are left out). Numbers come from the
sections named in brackets; all are aggregates.*

### In one paragraph

Dilution is predictable from public data: the single question "did the company file a share
offering in the last 12 months?" already gives an AUC of 0.71–0.74 in every test year. 409,601
rows (6,308 US companies) are eligible with an answer, and 1 in 10 of them dilute. The share
counts behind the answer are cleaned of typed-in errors (V2-23) and foreign filers are left out
(V2-27); rows with a stock split inside the window still need a check.

### What is clean

- **Coverage** [3]: steady 2012–2022; 86–89% of eligible rows have an answer.
- **The cleaning rules hold** [5]: no foreign filer has a market value; no market value is 0 or
  above \$5 trillion; no insider sales total above \$100 billion.
- **No leakage** [10]: every window is exactly 182 days after its row; no feature has an AUC
  above 0.95 or below 0.05.
- **Stable relations** [8]: none of the 15 strongest features moves to the other side of 0.5 in
  any year.
- **Purge** [10]: six-month windows cost 19,600–22,700 training rows (about 7%) before each test
  year. The 2022 test year has only 22,612 rows (January–June: later windows end after 2022).

### Who has no answer [5]

Of the rows passing the general rules, 94,513 have no answer because no new share count was filed
within 182 days: 78% of foreign-filer rows (they report shares once a year, in the 20-F; now left
out, V2-27), all price-only rows, and 11% of US rows.

### The answer [6]

- Base rate 9.78%; always saying 9.78% scores a Brier score of 0.0882 (the number to beat).
- About half of all companies (48.8%) dilute at least once.
- By sector: health care 23%, energy 13%, manufacturing 3.6%. By size: micro 17%, small 9.1%, mid
  4.6%, large 2.3%. Loss-making companies 19.5%, profitable ones 4.9%.

### Baseline and simple rules [7]

| rule | rows flagged | dilution if flagged | if not | AUC |
|---|---|---|---|---|
| an offering filed in the last 12 months | 35% | 21.5% | 3.5% | 0.73 |
| a shelf registration became effective in 12 months | 23% | 24.2% | 5.4% | 0.69 |
| shares already up 10% or more in 12 months | 16% | 27.4% | 6.4% | 0.66 |
| loss-making with less than a year of cash | 5% | 38.5% | 8.2% | 0.58 |

The offering rule holds in every test year (AUC 0.71–0.74): a strong *rule* level for 007.

### What relates to the answer [8]

- **Strongest single features** (AUC; below 0.5 means low values go with dilution): interest
  coverage 0.24 (only 42% of rows have it), offerings in 12 months 0.76 (top tenth dilutes 3.3×
  the base rate), R&D intensity 0.76 (24% have it), share growth 0.75 (3.5×), volatility
  0.70–0.72, stock-based pay to revenue 0.72, price level 0.30, market value 0.30.
- By family: financials, filing events, price and size carry the signal; insider activity is
  weaker (companies whose insiders sell are larger and dilute less); short volume and macro are
  useless here (|AUC − 0.5| ≤ 0.04).

### Share counts and splits [8]

- `share_growth_12m` reaches its cap of 20× (19 times as many shares): 1,401 rows above 10×.
- Rows with a stock split in the next six months (approximately: the company's sixth row after)
  dilute 25.0% of the time, against 9.5% for the rest (6,780 rows). Part of this is real
  (companies often do a reverse split and then sell shares); part may be a split that the share
  counts and the price series date differently, which would turn a split into a false dilution.

### Left for the validation round (named cases) [5, 8]

The split rows above; 22 rows (2 companies) above 50 billion shares; 12 rows of insider buys
above \$10 billion (one company, before it was listed).

### Advice for the dilution model (007)

- The answer is rare (10%): judge by AUC, log loss and the Brier score against 0.0882, and check
  calibration in the five bands (D47): each band's real dilution rate against its forecast.
- Rule level: the offering rule (AUC 0.73). A model is worth keeping only if it clearly beats it
  in every test year.
- Features: offerings and shelf filings, share growth, the financials (interest coverage, R&D,
  stock-based pay, cash, loss-making), volatility, price and size. Drop macro and short volume
  from the linear model.
- Missing values carry signal (no interest coverage usually means no interest expense: a small
  company without debt): add "was missing" flags in the linear model.

### Open questions for the researcher

- **Splits inside the window** (V2-25): check named companies with a split in the six months
  after a row and a "yes" answer, before the model trusts those rows.